# Notebook 08 — Fill Conventions: MTC vs Conservative

`BacktestEngine` supports two `fill_mode` options that determine the price used when
a trade executes on a signal-flip day:

| `fill_mode` | Entry fill | Exit fill | Interpretation |
|---|---|---|---|
| `"mtc"` (default) | Bar **open** | Bar **open** | Best-case: you act at the open |
| `"conservative"` | Bar **high** | Bar **low** | Worst-case: you bought the top, sold the bottom |

The gap between them measures the sensitivity of your strategy to execution quality.
A large spread means entry/exit timing matters a lot; a small spread means the signal
edge dominates fill slippage.

This matches the three return series in `BarBacktest`:
- `return_mark_to_close` ↔ `fill_mode="mtc"`
- `return_conservative` ↔ `fill_mode="conservative"`

In [ ]:
import pandas as pd
import plotly.graph_objects as go

from hailmary.data.providers import YahooFinanceProvider
from hailmary.models import TrendSignal
from hailmary.backtest import BacktestEngine, TrendSignalStrategy
from hailmary.backtest.signal_backtest import BarBacktest
from hailmary.viz.theme import PALETTE, apply_theme

In [ ]:
symbols = ["BTC-USD", "ETH-USD", "SOL-USD", "ADA-USD", "DOT-USD"]
start   = pd.Timestamp("2022-01-01")
end     = pd.Timestamp("2024-12-31")

signal = TrendSignal(ma_window=200)
yahoo  = YahooFinanceProvider()

fetch_start = start - pd.offsets.BDay(signal.warmup)
bars        = yahoo.get_bars(symbols, start=fetch_start, end=end)
signal_df   = signal.run(bars, trim_start=start)

print(f"Universe     : {symbols}")
print(f"Trading days : {signal_df.index.get_level_values('timestamp').nunique()}")

## 1. Run Both Fill Modes

In [ ]:
result_mtc = BacktestEngine(
    TrendSignalStrategy(signal_df),
    bars=bars,
    initial_capital=1_000_000,
    fill_mode="mtc",
).run(verbose=False)

result_con = BacktestEngine(
    TrendSignalStrategy(signal_df),
    bars=bars,
    initial_capital=1_000_000,
    fill_mode="conservative",
).run(verbose=False)

print(f"MTC NAV end          : ${result_mtc.nav.iloc[-1]:,.0f}")
print(f"Conservative NAV end : ${result_con.nav.iloc[-1]:,.0f}")
print(f"Execution drag       : ${result_mtc.nav.iloc[-1] - result_con.nav.iloc[-1]:,.0f}")

## 2. NAV Comparison

In [ ]:
def norm(nav: pd.Series) -> pd.Series:
    return nav / nav.iloc[0] * 100

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=result_mtc.nav.index, y=norm(result_mtc.nav),
    name="MTC (open fills)",
    line={"color": PALETTE["accent_blue"], "width": 2},
))
fig.add_trace(go.Scatter(
    x=result_con.nav.index, y=norm(result_con.nav),
    name="Conservative (high/low fills)",
    line={"color": PALETTE["accent_orange"], "width": 2},
))

apply_theme(fig, "Engine Path — MTC vs Conservative Fill Mode", height=460)
fig.update_layout(yaxis_title="NAV (base = 100)")
fig.show()

## 3. Summary Table

In [ ]:
rows = []
for label, res in [("MTC (open fills)", result_mtc), ("Conservative (high/low)", result_con)]:
    s = res.summary()
    rows.append({
        "Setup":        label,
        "Total Return": s["Total Return"],
        "Ann. Return":  s["Ann. Return"],
        "Sharpe":       s["Sharpe Ratio"],
        "Max DD":       s["Max Drawdown"],
        "Trades":       len(res.trade_log),
    })

pd.DataFrame(rows).set_index("Setup")